In [1]:
import numpy as np
import torch
import torch.nn as nn

np.random.seed(0)
torch.manual_seed(0)

# Environment
def step(s, a):
    ns = s + a + np.random.normal(0, 0.05)
    return ns, -ns**2

# Probabilistic Neural Network
class ProbNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 2)
        )

    def forward(self, s, a):
        mean, logvar = self.net(torch.cat([s, a], 1)).chunk(2, 1)
        return mean, logvar

    def loss(self, s, a, y):
        m, lv = self(s, a)
        return (.5 * ((y-m)**2 / torch.exp(lv) + lv)).mean()

# Ensemble
models = [ProbNet() for _ in range(3)]
opts = [torch.optim.Adam(m.parameters(), lr=.01) for m in models]

def train(S, A, Y):
    S = torch.tensor(S, dtype=torch.float32)[:,None]
    A = torch.tensor(A, dtype=torch.float32)[:,None]
    Y = torch.tensor(Y, dtype=torch.float32)[:,None]

    for m, opt in zip(models, opts):
        for _ in range(100):
            opt.zero_grad()
            loss = m.loss(S, A, Y)
            loss.backward()
            opt.step()

# Predict next state
def predict(s, a):
    m = models[np.random.randint(3)]
    s = torch.tensor([[s]], dtype=torch.float32)
    a = torch.tensor([[a]], dtype=torch.float32)

    with torch.no_grad():
        mean, logvar = m(s, a)
        return (mean + torch.exp(.5*logvar) *
                torch.randn_like(mean)).item()

# CEM Planner
def plan(s, H=5):
    mean, std = np.zeros(H), np.ones(H)

    for _ in range(3):
        actions = np.random.normal(mean, std, (100, H))
        scores = []

        for acts in actions:
            x, total = s, 0
            for a in acts:
                x = predict(x, a)
                total -= x**2
            scores.append(total)

        elite = actions[np.argsort(scores)[-10:]]
        mean, std = elite.mean(0), elite.std(0) + .001

    return mean[0]

# Initial training data
state = np.random.uniform(-5, 5)
S, A, Y = [], [], []

for _ in range(200):
    a = np.random.uniform(-1, 1)
    ns, _ = step(state, a)
    S.append(state)
    A.append(a)
    Y.append(ns)
    state = ns

train(np.array(S), np.array(A), np.array(Y))

# Planning
state = 5.0

for t in range(15):
    action = plan(state)
    ns, reward = step(state, action)

    print(f"t={t:2d} state={state:6.2f} "
          f"action={action:6.2f} reward={reward:6.2f}")

    S.append(state)
    A.append(action)
    Y.append(ns)

    state = ns
    train(np.array(S), np.array(A), np.array(Y))

t= 0 state=  5.00 action= -2.31 reward= -7.10
t= 1 state=  2.67 action= -1.88 reward= -0.63
t= 2 state=  0.79 action= -0.69 reward= -0.01
t= 3 state=  0.11 action= -0.10 reward= -0.00
t= 4 state= -0.00 action=  0.07 reward= -0.00
t= 5 state=  0.07 action= -0.11 reward= -0.02
t= 6 state= -0.12 action= -0.13 reward= -0.05
t= 7 state= -0.23 action=  0.06 reward= -0.04
t= 8 state= -0.21 action=  0.37 reward= -0.04
t= 9 state=  0.20 action= -0.17 reward= -0.00
t=10 state= -0.00 action= -0.02 reward= -0.00
t=11 state= -0.05 action=  0.05 reward= -0.00
t=12 state=  0.01 action= -0.08 reward= -0.03
t=13 state= -0.19 action=  0.21 reward= -0.00
t=14 state= -0.01 action=  0.16 reward= -0.01
